# Parsing de videojuegos y modelo de predicción

El CSV limpio contiene **474,415 juegos** y las columnas `slug_juego`, `es_game_jam`, `soporta_windows`, `soporta_mac`, `soporta_html5` y `es_demo`.

| Indicador | Casos | Porcentaje |
|---|---:|---:|
| Game Jam | 4,562 | 0.9616% |
| Windows | 276,755 | 58.3361% |
| Mac | 61,717 | 13.0091% |
| HTML5 | 386 | 0.0814% |
| Demo | 6,146 | 1.2955% |

**Alcance:** la fuente disponible es un CSV estructurado, no el volcado de texto crudo descrito. No tiene columnas explícitas de etiquetas/eventos; las casillas son heurísticas detectadas en los campos disponibles. El clasificador siguiente predice la etiqueta heurística `es_game_jam` a partir de las plataformas y `es_demo`, por lo que sirve como ejercicio exploratorio y no como ground truth.

In [1]:
from pathlib import Path

import pandas as pd

candidatos = [Path("dataset_juegos_limpio.csv"), Path("../dataset_juegos_limpio.csv")]
ruta_dataset = next((ruta for ruta in candidatos if ruta.exists()), None)
if ruta_dataset is None:
    raise FileNotFoundError("Ejecuta primero procesar_juegos.py para generar dataset_juegos_limpio.csv")

df = pd.read_csv(ruta_dataset)
print(f"Archivo: {ruta_dataset.resolve()}")
print(f"Registros: {len(df):,}")

resumen = []
for columna in ["es_game_jam", "soporta_windows", "soporta_mac", "soporta_html5", "es_demo"]:
    cantidad = int(df[columna].sum())
    resumen.append({"indicador": columna, "casos": cantidad, "porcentaje": 100 * cantidad / len(df)})
display(pd.DataFrame(resumen).set_index("indicador").round(4))

display(df.head(15))

columnas_sql = list(df.columns)
tipos_sql = {
    "slug_juego": "VARCHAR(255) PRIMARY KEY",
    "es_game_jam": "BOOLEAN NOT NULL",
    "soporta_windows": "BOOLEAN NOT NULL",
    "soporta_mac": "BOOLEAN NOT NULL",
    "soporta_html5": "BOOLEAN NOT NULL",
    "es_demo": "BOOLEAN NOT NULL",
}
print("CREATE TABLE juegos (")
print(",\n".join(f"    {columna} {tipos_sql[columna]}" for columna in columnas_sql))
print(");\n")
print("INSERT INTO juegos (" + ", ".join(columnas_sql) + ") VALUES")
filas_sql = []
for _, fila in df.head(15).iterrows():
    valores = ["'" + str(fila[columna]).replace("'", "''") + "'" if columna == "slug_juego" else ("TRUE" if bool(fila[columna]) else "FALSE") for columna in columnas_sql]
    filas_sql.append("    (" + ", ".join(valores) + ")")
print(",\n".join(filas_sql) + ";")

Archivo: C:\Users\Eric\OneDrive\Desktop\p\U\introduccion-progrmacion-cientifica-\dataset_juegos_limpio.csv
Registros: 474,415


,casos,porcentaje
indicador,,
es_game_jam,4562,0.9616
soporta_windows,276755,58.3361
soporta_mac,61717,13.0091
soporta_html5,386,0.0814
es_demo,6146,1.2955


,slug_juego,es_game_jam,soporta_windows,soporta_mac,soporta_html5,es_demo
0,dgeneration-hd,False,True,True,False,False
1,g-prime,False,True,True,False,False
2,land-sliders,False,False,False,False,False
3,pixel-gear,False,True,False,False,False
4,gods-and-idols,False,True,False,False,False
5,plague-venue,False,False,False,False,False
6,the-moon-sliver-itch,False,True,True,False,False
7,red-entity,False,True,True,False,False
8,hippiesvscops,False,True,False,False,False
9,they-came-through-the-floor,False,True,False,False,False


CREATE TABLE juegos (
    slug_juego VARCHAR(255) PRIMARY KEY,
    es_game_jam BOOLEAN NOT NULL,
    soporta_windows BOOLEAN NOT NULL,
    soporta_mac BOOLEAN NOT NULL,
    soporta_html5 BOOLEAN NOT NULL,
    es_demo BOOLEAN NOT NULL
);

INSERT INTO juegos (slug_juego, es_game_jam, soporta_windows, soporta_mac, soporta_html5, es_demo) VALUES
    ('dgeneration-hd', FALSE, TRUE, TRUE, FALSE, FALSE),
    ('g-prime', FALSE, TRUE, TRUE, FALSE, FALSE),
    ('land-sliders', FALSE, FALSE, FALSE, FALSE, FALSE),
    ('pixel-gear', FALSE, TRUE, FALSE, FALSE, FALSE),
    ('gods-and-idols', FALSE, TRUE, FALSE, FALSE, FALSE),
    ('plague-venue', FALSE, FALSE, FALSE, FALSE, FALSE),
    ('the-moon-sliver-itch', FALSE, TRUE, TRUE, FALSE, FALSE),
    ('red-entity', FALSE, TRUE, TRUE, FALSE, FALSE),
    ('hippiesvscops', FALSE, TRUE, FALSE, FALSE, FALSE),
    ('they-came-through-the-floor', FALSE, TRUE, FALSE, FALSE, FALSE),
    ('fading-light-descent', FALSE, TRUE, FALSE, FALSE, FALSE),
    ('the-book-

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

# La etiqueta es_game_jam está inferida por reglas; el modelo es exploratorio.
columnas_predictoras = [
    "soporta_windows",
    "soporta_mac",
    "soporta_html5",
    "es_demo",
]
X = df[columnas_predictoras].astype(int)
y = df["es_game_jam"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

modelo = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)
prediccion = modelo.predict(X_test)
probabilidad_jam = modelo.predict_proba(X_test)[:, 1]

metricas = pd.Series(
    {
        "accuracy": accuracy_score(y_test, prediccion),
        "balanced_accuracy": balanced_accuracy_score(y_test, prediccion),
        "precision": precision_score(y_test, prediccion, zero_division=0),
        "recall": recall_score(y_test, prediccion, zero_division=0),
        "f1": f1_score(y_test, prediccion, zero_division=0),
        "average_precision": average_precision_score(y_test, probabilidad_jam),
    },
    name="valor",
)
display(metricas.to_frame().round(4))

display(
    pd.DataFrame(
        confusion_matrix(y_test, prediccion),
        index=["real_no_jam", "real_jam"],
        columns=["pred_no_jam", "pred_jam"],
    )
)

nuevo_juego = pd.DataFrame(
    [{"soporta_windows": 1, "soporta_mac": 0, "soporta_html5": 0, "es_demo": 0}]
)
probabilidad = modelo.predict_proba(nuevo_juego[columnas_predictoras])[0, 1]
print(f"Probabilidad estimada de Game Jam para el ejemplo: {probabilidad:.2%}")
print("Predicción:", "Game Jam" if probabilidad >= 0.5 else "No Game Jam")

,valor
accuracy,0.4269
balanced_accuracy,0.5364
precision,0.0108
recall,0.6480
f1,0.0213
average_precision,0.0106


,pred_no_jam,pred_jam
real_no_jam,39912,54059
real_jam,321,591


Probabilidad estimada de Game Jam para el ejemplo: 52.72%
Predicción: Game Jam


## Conclusión del modelo

El modelo **todavía no sirve para decidir automáticamente** si un juego pertenece a una Game Jam. Aunque encuentra aproximadamente **65 de cada 100** juegos que la tabla marca como Game Jam (recall = 64.8%), se equivoca mucho al dar avisos: **solo cerca de 1 de cada 100 juegos que predice como Game Jam realmente tiene esa etiqueta** (precision = 1.08%). En otras palabras, casi todos sus avisos son falsos positivos.

La probabilidad de **52.72%** del ejemplo Windows-only tampoco debe tomarse como una conclusión fiable. El modelo apenas mejora una selección al azar y, además, aprende etiquetas creadas automáticamente mediante reglas, no confirmadas por datos oficiales. Para obtener predicciones útiles se necesitan etiquetas reales de Game Jam y más información de cada juego, como eventos verificados, descripción y fecha de participación.